# Dijkstra's algorithm and priority queues: notebook

The room of RO-009 and RO-032, now with moves of different lengths: the 0.5 m grid with diagonal moves
(8-connected) and the waypoint graph with its edge lengths. The geometry lives in `images/roommap.py`, the searches
and the heap in `images/roomsearch.py`; here we check every number the Note quotes. Every run is deterministic.

In [1]:
import sys, math, heapq
sys.path.insert(0, 'images')
import roomsearch as rs

## 1. Diagonal moves (Note, Sections 1 and 2)

In [2]:
print('diagonal move:', math.sqrt(0.5 ** 2 + 0.5 ** 2))
edges8 = sorted({tuple(sorted((c, n))) for c in rs.FREE for n, _ in rs.neighbours(c, rs.STEPS8)})
print('8-connected edges:', len(edges8), '(4-connected: 114)')
b = rs.bfs(rs.STEPS8)
diag = lambda p: sum(1 for a, c in zip(p, p[1:]) if a[0] != c[0] and a[1] != c[1])
print('BFS: moves', len(b['path']) - 1, ' diagonals', diag(b['path']), ' length', round(rs.length(b['path']), 4),
      ' expansions', len(b['visited']))
d = rs.dijkstra(lambda c: rs.neighbours(c, rs.STEPS8), rs.START, rs.GOAL)
print('Dijkstra: moves', len(d['path']) - 1, ' diagonals', diag(d['path']), ' length', round(d['cost'], 4),
      ' settled', len(d['settled']))
print('BFS route', b['path']); print('Dijkstra route', d['path'])

diagonal move: 0.7071067811865476
8-connected edges: 190 (4-connected: 114)
BFS: moves 6  diagonals 4  length 3.8284  expansions 54
Dijkstra: moves 6  diagonals 2  length 3.4142  settled 54
BFS route [(4, 2), (3, 3), (3, 4), (4, 5), (5, 6), (6, 6), (7, 7)]
Dijkstra route [(4, 2), (4, 3), (4, 4), (4, 5), (5, 6), (6, 6), (7, 7)]


## 2. The waypoint graph: breadth-first search ignores the lengths (Note, Section 2.2)

In [3]:
for name, nb_ in [('alphabetical', rs.wp_nbrs), ('reverse', lambda v: rs.wp_nbrs(v)[::-1])]:
    p = rs.bfs(start='S', goal='G', nbrs=nb_)['path']
    print(name, 'order:', '-'.join(p), round(sum(rs.wp_dist(a, c) for a, c in zip(p, p[1:])), 3), 'm')

alphabetical order: S-D-G 3.505 m
reverse order: S-P-G 5.022 m


## 3. Dijkstra's algorithm on the waypoint graph (Note, Section 3)

In [4]:
w = rs.dijkstra(rs.wp_nbrs, 'S', 'G', stop_at_goal=False)
for s in w['snaps']:
    print(f"settle {s['cur']} at {s['cost']:.3f}; queue {[(v, round(c, 3)) for c, v in s['queue']]};",
          'lowered:', [(v, round(a, 3), round(b_, 3)) for v, a, b_ in s['improved']])
print('route to G:', rs.path_to(w['parent'], 'G'), round(w['g']['G'], 3))
u = rs.dijkstra(rs.wp_nbrs, 'S', 'G')
print('stopped at the goal (uniform-cost search): settled', u['settled'])

settle S at 0.000; queue [('A', 1.0), ('P', 2.236), ('D', 2.5)]; lowered: []
settle A at 1.000; queue [('P', 2.236), ('D', 2.5), ('B', 3.0)]; lowered: []
settle P at 2.236; queue [('D', 2.5), ('B', 3.0), ('G', 5.022)]; lowered: []
settle D at 2.500; queue [('B', 3.0), ('G', 3.505), ('C', 4.562)]; lowered: [('G', 5.022, 3.505)]
settle B at 3.000; queue [('G', 3.505), ('C', 4.562)]; lowered: []
settle G at 3.505; queue [('C', 4.562)]; lowered: []
settle C at 4.562; queue []; lowered: []
route to G: ['S', 'D', 'G'] 3.505
stopped at the goal (uniform-cost search): settled ['S', 'A', 'P', 'D', 'B', 'G']


## 4. Dijkstra's algorithm on the grid (Note, Section 4)

In [5]:
print('settled before G:', len(d['settled']), ' pops:', d['snaps'][-1]['pops'])
print('pushes:', 1 + sum(len(s['added']) + len(s['improved']) for s in d['snaps']),
      ' entries lowered:', sum(len(s['improved']) for s in d['snaps']))
print('largest queue:', max(len(s['queue']) for s in d['snaps']))
full = rs.dijkstra(lambda c: rs.neighbours(c, rs.STEPS8), rs.START, rs.GOAL, stop_at_goal=False)
print('without stopping: settled', len(full['settled']), ' farthest', round(max(full['g'].values()), 3), 'm')
ties = sorted(round(v, 4) for c, v in full['g'].items() if abs(v - d['cost']) < 1e-9)
print('cells at the same distance as G:', len(ties))

settled before G: 54  pops: 54
pushes: 59  entries lowered: 0
largest queue: 16
without stopping: settled 76  farthest 8.121 m
cells at the same distance as G: 3


## 5. The cost of a priority queue (Note, Section 5)

In [6]:
N = 10 ** 6
print('list: N^2 =', f'{N ** 2:.0e}', '->', N ** 2 / 1e8, 's at 1e8 per second')
print('heap: N log2 N =', f'{N * math.log2(N):.2e}', '->', round(N * math.log2(N) / 1e8, 2), 's')
for n in (1, 3, 7, 15, 10 ** 6):
    print(n, 'entries need', math.floor(math.log2(n)) + 1, 'levels')
print('2^20 =', 2 ** 20)
V, E = 10 ** 6, 4 * 10 ** 6
print('Dijkstra with a heap: (V + E) log2 V =', f'{(V + E) * math.log2(V):.2e}', '; with a list: V^2 =', f'{V ** 2:.0e}')

list: N^2 = 1e+12 -> 10000.0 s at 1e8 per second
heap: N log2 N = 1.99e+07 -> 0.2 s
1 entries need 1 levels
3 entries need 2 levels
7 entries need 3 levels
15 entries need 4 levels
1000000 entries need 20 levels
2^20 = 1048576
Dijkstra with a heap: (V + E) log2 V = 9.97e+07 ; with a list: V^2 = 1e+12


## 6. The heap of the waypoint run, step by step (Note, Section 5.2)

In [7]:
h = rs.MinHeap()
SP = rs.wp_dist('S', 'P')
ops = [('push', (0.0, 'S')), ('pop', None), ('push', (1.0, 'A')), ('push', (2.5, 'D')), ('push', (SP, 'P')),
       ('pop', None), ('push', (3.0, 'B')), ('pop', None), ('push', (SP + rs.wp_dist('P', 'G'), 'G')), ('pop', None),
       ('push', (2.5 + rs.wp_dist('D', 'C'), 'C')), ('push', (2.5 + rs.wp_dist('D', 'G'), 'G')), ('pop', None),
       ('pop', None)]
for op, item in ops:
    n0 = len(h.steps)
    out = h.push(item) if op == 'push' else h.pop()
    steps = [(w_, [(v, round(c, 3)) for c, v in a]) for w_, a, _ in h.steps[n0:]]
    print(op, item and (item[1], round(item[0], 3)), '-> removed' if op == 'pop' else '', out and (out[1], round(out[0], 3)))
    for st in steps:
        print('    ', st)

push ('S', 0.0)  None
     ('place', [('S', 0.0)])
pop None -> removed ('S', 0.0)
push ('A', 1.0)  None
     ('place', [('A', 1.0)])
push ('D', 2.5)  None
     ('place', [('A', 1.0), ('D', 2.5)])
push ('P', 2.236)  None
     ('place', [('A', 1.0), ('D', 2.5), ('P', 2.236)])
pop None -> removed ('A', 1.0)
     ('last to root', [('P', 2.236), ('D', 2.5)])
push ('B', 3.0)  None
     ('place', [('P', 2.236), ('D', 2.5), ('B', 3.0)])
pop None -> removed ('P', 2.236)
     ('last to root', [('B', 3.0), ('D', 2.5)])
     ('sift down', [('D', 2.5), ('B', 3.0)])
push ('G', 5.022)  None
     ('place', [('D', 2.5), ('B', 3.0), ('G', 5.022)])
pop None -> removed ('D', 2.5)
     ('last to root', [('G', 5.022), ('B', 3.0)])
     ('sift down', [('B', 3.0), ('G', 5.022)])
push ('C', 4.562)  None
     ('place', [('B', 3.0), ('G', 5.022), ('C', 4.562)])
push ('G', 3.505)  None
     ('place', [('B', 3.0), ('G', 5.022), ('C', 4.562), ('G', 3.505)])
     ('sift up', [('B', 3.0), ('G', 3.505), ('C', 4.562), 

Python's own `heapq` module keeps the same kind of heap in a plain list; Dijkstra's algorithm in
`roomsearch.py` uses it. The Python box of Section 5.3:

In [8]:
def dijkstra(start, goal, neighbours):
    dist, parent = {start: 0.0}, {start: None}
    queue, settled = [(0.0, start)], set()
    while queue:
        d_, v = heapq.heappop(queue)
        if v in settled:
            continue                          # a stale entry
        settled.add(v)
        if v == goal:
            break
        for n, cost in neighbours(v):
            if d_ + cost < dist.get(n, math.inf):
                dist[n], parent[n] = d_ + cost, v
                heapq.heappush(queue, (dist[n], n))
    return dist[goal]

print(round(dijkstra('S', 'G', rs.wp_nbrs), 3))
print(round(dijkstra(rs.START, rs.GOAL, lambda c: rs.neighbours(c, rs.STEPS8)), 3))

3.505
3.414
